# LoRA text-to-SQL on a free Colab GPU

Runs the whole experiment: baselines, LoRA fine-tune, and the comparison table.

**Before you start:** Runtime > Change runtime type > **T4 GPU**.

In [ ]:
# Point this at your copy of the repository.
REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/lora-text-to-sql.git"

!git clone -q $REPO_URL repo
%cd repo
!pip install -q -e .
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Plumbing check (about 30 seconds)
A tiny random model is trained on synthetic data. This proves every step runs before you spend GPU time.

In [ ]:
!sqltune smoke

## 2. Data: download `b-mc2/sql-create-context`, de-duplicate, split

In [ ]:
!sqltune prepare --n-val 500 --n-test 1000

## 3. Fine-tune with LoRA
The defaults train on 10,000 examples for one epoch. `train_seconds` in `outputs/lora/train_summary.json` records how long it took.

In [ ]:
!sqltune train --max-train-examples 10000 --batch-size 16

## 4. Compare zero-shot, few-shot and LoRA on the same 1,000 test questions

In [ ]:
!sqltune evaluate --modes zero_shot few_shot lora

In [ ]:
from IPython.display import Markdown, display

display(Markdown(open("results/comparison.md").read()))

## 5. Training curve

In [ ]:
import json

import matplotlib.pyplot as plt

hist = json.load(open("outputs/lora/train_summary.json"))["log_history"]
train = [(h["step"], h["loss"]) for h in hist if "loss" in h]
val = [(h["step"], h["eval_loss"]) for h in hist if "eval_loss" in h]
plt.plot(*zip(*train, strict=True), label="train")
plt.plot(*zip(*val, strict=True), "o-", label="validation")
plt.xlabel("step")
plt.ylabel("loss")
plt.title("LoRA fine-tuning")
plt.legend()
plt.savefig("results/loss_curve.png", dpi=120, bbox_inches="tight")
plt.show()

## 6. Inspect failures
Reading the wrong predictions shows what the model actually struggles with.

In [ ]:
import json

res = json.load(open("results/comparison.json"))["lora"]["per_example"]
wrong = [r for r in res if not r["exec"]][:15]
for r in wrong:
    print("Q:   ", r["question"])
    print("gold:", r["gold"])
    print("pred:", r["pred"])
    print()

## 7. Save the results
Download `results/` and commit it to the repo so the README table shows real numbers. To share the adapter, fill in `MODEL_CARD.md` and run:

```python
from huggingface_hub import login; login()
from peft import PeftModel
from transformers import AutoModelForCausalLM
base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
PeftModel.from_pretrained(base, "outputs/lora/adapter").push_to_hub("YOUR_HF_USERNAME/qwen2.5-0.5b-text-to-sql-lora")
```

In [ ]:
!zip -qr results.zip results outputs/lora/train_summary.json && echo 'download results.zip from the file browser'